# B2-022-probabilistic-latent-models — Practice p21 — Solution

**Type:** scenario · **Difficulty:** core · **Concepts:** multivariate-gaussian, autoencoder, variational-autoencoder

*55 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

## Solution

Parameter counts used below: a diagonal Gaussian in $d$ dimensions has $2d$ parameters; a full-covariance Gaussian has $d+d(d+1)/2$. The full-covariance MLE $\hat\Sigma=\frac1n\sum(x-\bar x)(x-\bar x)^\top$ exists only if it is nonsingular, which needs $n\ge d+1$ (and the centered rows to span $\mathbb R^d$); the diagonal MLE needs only that each coordinate is non-constant (p13).

### (I) Anomaly scores from 8 correlated sensors

**Decision:** full covariance.
**Justification:** With $d=8$ the full model has $8+36=44$ parameters versus $16$ for the diagonal, and $n=400\gg d+1=9$, so the full MLE exists and is well estimated (about 9 rows per parameter). A correlation of $0.9$ between sensors 1 and 2 means the diagonal model would treat a reading with sensor 1 high and sensor 2 low as typical, since each coordinate alone is ordinary, while the full covariance's Mahalanobis term flags exactly that broken correlation. The goal is a log-density score, so modelling the dependence is the point.
**Falsifying held-out check:** if the full-covariance model's mean held-out log-density (on rows never used to fit it) is not higher than the diagonal model's, the recommendation is wrong.

### (II) Wide, short table

**Decision:** diagonal covariance.
**Justification:** The full model would need $50+50\cdot51/2=1325$ parameters from $n=30$ rows, and since $n=30<d+1=51$ the centered rows span at most a 29-dimensional subspace, so $\hat\Sigma$ is singular: $\log\det\hat\Sigma=-\infty$, the likelihood is unbounded (no MLE exists), and $\hat\Sigma^{-1}$ needed for the score does not exist. The diagonal model has $100$ parameters, and its MLE exists whenever each column is non-constant across the $30$ rows.
**Falsifying held-out check:** if the diagonal model's held-out log-densities are not finite or are worse than those of a simpler pooled-variance (isotropic) Gaussian fitted on the same train rows, the choice is wrong.

### (III) Dashboard compression

**Decision:** (deterministic) autoencoder with a 2-D code.
**Justification:** The goal is only compress-to-2 and reconstruct accurately; no sampling is ever needed, so the VAE's KL term, which pulls codes toward $N(0,I)$ and injects noise into the code during training, only costs reconstruction accuracy. An undercomplete autoencoder directly minimizes reconstruction error through the 2-number bottleneck.
**Falsifying held-out check:** if a VAE of the same size (using its posterior mean as the code) achieves lower held-out reconstruction MSE than the autoencoder, the recommendation is wrong.

### (IV) Synthetic data

**Decision:** VAE.
**Justification:** Generation needs a known distribution to sample codes from; the VAE's KL term keeps the aggregate posterior close to the prior $N(0,I)$, so decoding prior draws gives realistic readings, while an autoencoder's code space has no such distribution and has holes where decoded points are unrealistic. The two operations are **prior sampling** (decode $z\sim N(0,I)$) and **latent interpolation** (decode $(1-t)\mu_a+t\mu_b$ between the posterior means of the two recorded operating states).
**Falsifying held-out check:** if decoded prior samples fail to match held-out reading statistics (per-feature mean and spread, and coverage of the known operating modes) at least as well as an autoencoder with randomly sampled codes, the recommendation is wrong.

### (V) Linear structure only

**Decision:** PCA (top-2 principal components of the centered training data).
**Justification:** The best linear 2-number compression in squared error is given exactly in closed form by PCA (Eckart–Young), so neither iterative training nor a nonlinear/stochastic model is needed. A linear autoencoder trained with MSE can at best recover the same subspace, and its trained decoder subspace should be compared against the PCA subspace through the orthogonal projectors $P_{W_{ae}}$ and $P_{W_{pca}}$ (as in p18), not weight by weight.
**Falsifying held-out check:** if the held-out reconstruction MSE of the 2-component PCA projection is noticeably worse than that of a (nonlinear) autoencoder or VAE with a 2-D code, the "linear structure only" premise and the PCA choice are wrong.

In [ ]:
def gaussian_param_counts(d):
    return {"diagonal": 2 * d, "full": d + d * (d + 1) // 2}

def full_mle_can_exist(n, d):
    return n >= d + 1

DECISIONS = {
    "I": "full covariance",
    "II": "diagonal covariance",
    "III": "autoencoder",
    "IV": "VAE (prior sampling + latent interpolation)",
    "V": "PCA (compare linear AE via projectors)",
}
counts_8, counts_50 = gaussian_param_counts(8), gaussian_param_counts(50)
print(counts_8, counts_50, full_mle_can_exist(400, 8), full_mle_can_exist(30, 50))

### Answer check

In [ ]:
assert counts_8 == {"diagonal": 16, "full": 44}
assert counts_50 == {"diagonal": 100, "full": 1325}
assert full_mle_can_exist(400, 8) and not full_mle_can_exist(30, 50)
assert set(DECISIONS) == {"I", "II", "III", "IV", "V"}
assert DECISIONS["III"] == "autoencoder" and DECISIONS["IV"].startswith("VAE")